In [ ]:
from itertools import permutations
class Container:
    def __init__(self,length = 590,width = 235, height = 239):
        self.length = length
        self.width = width
        self.height = height
        self.placed_boxes = []
        self.free_spaces = [
            {"x":0,
             "y":0,
             "z":0,
             "L":length,
             "W":width,
             "H":height
            }
        ]
        self.unplaced_boxes = []

        pass
    # returns all possible orientations of a box, example (30,40,20) =>  ( (30,40,20),(20,40,30) ect... )
    def get_orientations(self,box):
        dims = (box["l"],box["w"],box["h"])
        return list(set(permutations(dims)))

    #checks wether a given orientation of a box fits in a given space
    def fits(self,l,w,h,space):
         return l <= space["L"] and w <= space["W"] and h <= space["H"]


    #use the Deppest bottom left to chose the best placement for a box iterating throught 
    # the free spaces and all the possible orientations for the box
    #returns the choosen space and the choosen orientaton for the box if it can fit
    def find_best_placement_dblf(self, box):

        best_choice = None

        #  try all orientations
        for (l, w, h) in self.get_orientations(box):

            #  try all free spaces
            for space in self.free_spaces:

                #  check feasibility
                if self.fits(l, w, h, space):

                    # DBLF priority
                    score = (space["z"], space["x"], space["y"])

                    #  keep best candidate
                    if best_choice is None or score < best_choice["score"]:

                        best_choice = {
                            "space": space,
                            "dims": (l, w, h),
                            "score": score
                        }

        return best_choice

    
    #after finding where to place the box, we place it, then we remove the free space used by splitting it into smaller free spaces
    def place_box(self,box,choice):
        if choice is None:
            self.unplaced_boxes.append({
            "id": box["id"],
            "reason": "no valid placement found"
            })
            return False  # cannot place

        space = choice["space"]
        l, w, h = choice["dims"]

        x, y, z = space["x"], space["y"], space["z"]

        placed = {
        "id": box["id"],
        "x": x,
        "y": y,
        "z": z,
        "l": l,
        "w": w,
        "h": h
        }

        self.placed_boxes.append(placed)

    # remove used space
        self.free_spaces.remove(space)

    # split space
        new_spaces = self.split_space(space, l, w, h, x, y, z)

        self.free_spaces.extend(new_spaces)

        return True

    # we split the free space used to other to smaller free spaces
    def split_space(self, space, l, w, h, x, y, z):

        new_spaces = []

        # Right
        if space["L"] > l:
            new_spaces.append({
                "x": x + l,
                "y": y,
                "z": z,
                "L": space["L"] - l,
                "W": space["W"],
                "H": space["H"]
            })

        # Front
        if space["W"] > w:
            new_spaces.append({
                "x": x,
                "y": y + w,
                "z": z,
                "L": space["L"],        # keep full length
                "W": space["W"] - w,
                "H": space["H"]
            })

        # Top
        if space["H"] > h:
            new_spaces.append({
                "x": x,
                "y": y,
                "z": z + h,
                "L": space["L"],        # full footprint
                "W": space["W"],
                "H": space["H"] - h
            })

        return new_spaces